# Scientific Workbench: Signal Processing & Recurrent Dynamics

## 1. Interactive Signal Laboratory
Adjust parameters to visualize the complexity of the composite signal $S_{total}$.

In [ ]:
import os
import sys

sys.path.append(os.path.abspath(".."))

import matplotlib.pyplot as plt
import seaborn as sns
from src.sdk.dataset import SignalDataset
from src.sdk.analysis_utils import calculate_psd, get_lstm_gates
from src.models.lstm import LSTMFilter
from src.sdk.gatekeeper import APIGatekeeper

noise_lvl = 0.4  # @param {type:"slider", min:0, max:1, step:0.1}
ds = SignalDataset(num_samples=1, window_size=1000, noise_level=noise_lvl)
x, y = ds[0]

plt.figure(figsize=(12, 4))
plt.plot(x[:, 4], label="Composite Noisy Input", color="gray", alpha=0.5)
plt.plot(y, label="1Hz Ground Truth", color="blue")
plt.title(f"Signal Lab: Noise Level {noise_lvl}")
plt.legend()
plt.show()

## 2. Spectral Analysis (FFT)
We use Power Spectral Density (PSD) to prove the network is nulling interfering frequencies.

In [ ]:
model = LSTMFilter(hidden_dim=32)
gk = APIGatekeeper(model)
pred = gk.run_inference(x.unsqueeze(0))

f_in, p_in = calculate_psd(x[:, 4])
f_out, p_out = calculate_psd(pred[0])

plt.figure(figsize=(10, 4))
plt.semilogy(f_in, p_in, label="Input Spectrum (Noisy Sum)", color="gray")
plt.semilogy(f_out, p_out, label="Output Spectrum (LSTM Filtered)", color="red")
plt.xlim(0, 15)
plt.title("Spectral Denoising Verification (FFT)")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Power/Freq (dB/Hz)")
plt.legend()
plt.show()

## 3. Gating Dynamics Heatmap
Visualizing the internal activation of the Forget, Input, and Output gates.

In [ ]:
gates = get_lstm_gates(model, x.unsqueeze(0))
plt.figure(figsize=(10, 3))
sns.heatmap(gates["forget"][:100].T, cmap="magma")
plt.title("LSTM Forget Gate Activation (LEC Stability Check)")
plt.xlabel("Timesteps")
plt.ylabel("Hidden Dim")
plt.show()